In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visualization style
sns.set_theme(style="whitegrid")

# Define the path to your data (using relative paths from the notebooks folder)
data_dir = '../data/'
dos_file = os.path.join(data_dir, 'DoS_dataset.csv')

# The HCRL dataset typically lacks column headers, so we define them based on CAN protocol standard
# Timestamp, CAN ID, DLC (Data Length Code), 8 Data Bytes, and the Attack Label (Flag)
columns = ['Timestamp', 'CAN_ID', 'DLC', 'Data0', 'Data1', 'Data2', 'Data3', 'Data4', 'Data5', 'Data6', 'Data7', 'Label']

print("Loading a sample of the DoS Dataset...")
# Read only the first 500,000 rows to save memory during exploration
df_dos = pd.read_csv(dos_file, header=None, names=columns, nrows=500000)

# Display basic info
print(f"Dataset Shape: {df_dos.shape}")
print("\nClass Distribution (Normal 'R' vs Attack 'T'):")
print(df_dos['Label'].value_counts())

# Preview the first few rows
display(df_dos.head())

Loading a sample of the DoS Dataset...
Dataset Shape: (500000, 12)

Class Distribution (Normal 'R' vs Attack 'T'):
Label
R    391407
T    104728
Name: count, dtype: int64


,Timestamp,CAN_ID,DLC,Data0,Data1,Data2,Data3,Data4,Data5,Data6,Data7,Label
0,1.478198e+09,0316,8,05,21,68,09,21,21,00,6f,R
1,1.478198e+09,018f,8,fe,5b,00,00,00,3c,00,00,R
2,1.478198e+09,0260,8,19,21,22,30,08,8e,6d,3a,R
3,1.478198e+09,02a0,8,64,00,9a,1d,97,02,bd,00,R
4,1.478198e+09,0329,8,40,bb,7f,14,11,20,00,14,R


In [2]:
import math
from collections import Counter

print("Starting Feature Engineering...")

# 1. Label Encoding: The dataset uses 'R' for Normal and 'T' for Attack.
# We need numerical labels for our ML models.
df_dos['Is_Attack'] = df_dos['Label'].apply(lambda x: 1 if x == 'T' else 0)

# 2. Global Timing Intervals: Time difference between consecutive messages on the bus
df_dos['Time_Diff'] = df_dos['Timestamp'].diff().fillna(0)

# 3. Per-ID Timing Intervals: Time difference between messages of the SAME CAN_ID
# Attackers often flood specific IDs, dropping the time difference drastically.
print("Calculating per-ID timing features...")
df_dos['Time_Diff_Per_ID'] = df_dos.groupby('CAN_ID')['Timestamp'].diff().fillna(0)

# 4. Payload Entropy: Calculates the randomness of the data payload.
def calculate_entropy(row):
    # Gather data bytes, dropping NaNs (sometimes DLC is less than 8)
    data_bytes = [val for val in row[['Data0', 'Data1', 'Data2', 'Data3', 'Data4', 'Data5', 'Data6', 'Data7']] if pd.notna(val)]
    if not data_bytes:
        return 0
    
    # Calculate Shannon Entropy
    byte_counts = Counter(data_bytes)
    entropy = 0
    total_len = len(data_bytes)
    for count in byte_counts.values():
        p_x = count / total_len
        entropy += - p_x * math.log2(p_x)
    return entropy

print("Calculating payload entropy (this might take a few seconds)...")
df_dos['Payload_Entropy'] = df_dos.apply(calculate_entropy, axis=1)

# Drop the original categorical data columns since we've extracted the entropy
features_df = df_dos[['Timestamp', 'CAN_ID', 'DLC', 'Time_Diff', 'Time_Diff_Per_ID', 'Payload_Entropy', 'Is_Attack']]

print("\nFeature Engineering Complete! Here is the new dataset structure:")
display(features_df.head(10))

Starting Feature Engineering...
Calculating per-ID timing features...
Calculating payload entropy (this might take a few seconds)...

Feature Engineering Complete! Here is the new dataset structure:


,Timestamp,CAN_ID,DLC,Time_Diff,Time_Diff_Per_ID,Payload_Entropy,Is_Attack
0,1.478198e+09,0316,8,0.000000,0.0,2.405639,0
1,1.478198e+09,018f,8,0.000209,0.0,1.548795,0
2,1.478198e+09,0260,8,0.000228,0.0,3.000000,0
3,1.478198e+09,02a0,8,0.000232,0.0,2.750000,0
4,1.478198e+09,0329,8,0.000237,0.0,2.750000,0
5,1.478198e+09,0545,8,0.000241,0.0,1.061278,0
6,1.478198e+09,0002,8,0.002743,0.0,1.548795,0
7,1.478198e+09,0153,8,0.000235,0.0,1.750000,0
8,1.478198e+09,02c0,8,0.000623,0.0,0.543564,0
9,1.478198e+09,0130,8,0.000239,0.0,2.750000,0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import time

print("Preparing data for modeling...")

# Ensure data is sorted by Timestamp for temporal splitting
features_df = features_df.sort_values(by='Timestamp')

# Features and Target
# We drop 'Timestamp' and 'CAN_ID' for the model, focusing on the engineered behavior
X = features_df[['DLC', 'Time_Diff', 'Time_Diff_Per_ID', 'Payload_Entropy']]
y = features_df['Is_Attack']

# Temporal Split: 80% train, 20% test. shuffle=False is critical for time-series!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples\n")

# --- 1. Supervised Learning: XGBoost ---
print("Training Supervised Model (XGBoost)...")
start_time = time.time()
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start_time

print(f"XGBoost trained in {xgb_time:.2f} seconds")
xgb_preds = xgb_model.predict(X_test)

# --- 2. Unsupervised Learning: Isolation Forest ---
print("\nTraining Unsupervised Baseline (Isolation Forest)...")
start_time = time.time()
# Contamination is the estimated proportion of outliers in the dataset
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
# We only train IF on normal data (y==0) to establish the baseline
X_train_normal = X_train[y_train == 0]
iso_forest.fit(X_train_normal)
iso_time = time.time() - start_time

print(f"Isolation Forest trained in {iso_time:.2f} seconds")
# Predict returns 1 for inliers (normal) and -1 for outliers (attack). Map it to 0 and 1.
iso_preds_raw = iso_forest.predict(X_test)
iso_preds = np.where(iso_preds_raw == 1, 0, 1)

# --- 3. Evaluation ---
print("\n=== XGBoost Performance ===")
print(classification_report(y_test, xgb_preds, target_names=['Normal', 'Attack']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))

print("\n=== Isolation Forest Performance ===")
print(classification_report(y_test, iso_preds, target_names=['Normal', 'Attack']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, iso_preds))

Preparing data for modeling...
Training set: 400000 samples
Testing set:  100000 samples

Training Supervised Model (XGBoost)...


d:\Assignments\automotive_ids_project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [18:41:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost trained in 5.33 seconds

Training Unsupervised Baseline (Isolation Forest)...
Isolation Forest trained in 4.22 seconds

=== XGBoost Performance ===
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     80926
      Attack       1.00      1.00      1.00     19074

    accuracy                           1.00    100000
   macro avg       1.00      1.00      1.00    100000
weighted avg       1.00      1.00      1.00    100000

Confusion Matrix:
[[80923     3]
 [   12 19062]]

=== Isolation Forest Performance ===
              precision    recall  f1-score   support

      Normal       0.84      0.95      0.89     80926
      Attack       0.52      0.25      0.33     19074

    accuracy                           0.81    100000
   macro avg       0.68      0.60      0.61    100000
weighted avg       0.78      0.81      0.78    100000

Confusion Matrix:
[[76515  4411]
 [14352  4722]]


In [4]:
print("Implementing Windowed Temporal Analysis...")

window_size = 100

# 1. Calculate rolling statistics on our engineered features
print(f"Calculating rolling means with window size = {window_size}...")
features_df['Time_Diff_Rolling_Mean'] = features_df['Time_Diff'].rolling(window=window_size).mean().fillna(0)
features_df['Time_Diff_Per_ID_Rolling_Mean'] = features_df['Time_Diff_Per_ID'].rolling(window=window_size).mean().fillna(0)
features_df['Payload_Entropy_Rolling_Mean'] = features_df['Payload_Entropy'].rolling(window=window_size).mean().fillna(0)

# 2. Window-based labeling: If any message in the window is an attack, flag the whole window
features_df['Window_Is_Attack'] = features_df['Is_Attack'].rolling(window=window_size).max().fillna(0)

# 3. Prepare data for the windowed model
X_win = features_df[['Time_Diff_Rolling_Mean', 'Time_Diff_Per_ID_Rolling_Mean', 'Payload_Entropy_Rolling_Mean']]
y_win = features_df['Window_Is_Attack']

# Temporal Split
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_win, y_win, test_size=0.2, shuffle=False)

# 4. Train XGBoost on the windowed data
print("Training XGBoost on Windowed Features...")
start_time_w = time.time()
xgb_model_win = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model_win.fit(X_train_w, y_train_w)
xgb_time_w = time.time() - start_time_w

# 5. Evaluate and check Detection Latency
xgb_preds_w = xgb_model_win.predict(X_test_w)

print(f"\nWindowed Model trained in {xgb_time_w:.2f} seconds")
print("\n=== Windowed XGBoost Performance ===")
print(classification_report(y_test_w, xgb_preds_w, target_names=['Normal Window', 'Attack Window']))

# Calculate average inference time per window to assess real-time feasibility
inference_start = time.time()
_ = xgb_model_win.predict(X_test_w.iloc[:1000]) # Predict 1000 windows
inference_end = time.time()
avg_inference_ms = ((inference_end - inference_start) / 1000) * 1000

print(f"\nModel inference time: {avg_inference_ms:.4f} ms per window")
print("Target: <20-30ms for real-time processing of 100-message windows.")

Implementing Windowed Temporal Analysis...
Calculating rolling means with window size = 100...
Training XGBoost on Windowed Features...


d:\Assignments\automotive_ids_project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [18:44:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Windowed Model trained in 1.33 seconds

=== Windowed XGBoost Performance ===
               precision    recall  f1-score   support

Normal Window       1.00      1.00      1.00     64289
Attack Window       1.00      1.00      1.00     35711

     accuracy                           1.00    100000
    macro avg       1.00      1.00      1.00    100000
 weighted avg       1.00      1.00      1.00    100000


Model inference time: 0.0000 ms per window
Target: <20-30ms for real-time processing of 100-message windows.


In [6]:
import pickle
import os

# Ensure the models directory exists
os.makedirs('../models', exist_ok=True)

print("Saving CAN IDS models as .pkl for final deliverable...")

# Saving the windowed XGBoost model
with open('../models/can_xgboost_windowed.pkl', 'wb') as f:
    pickle.dump(xgb_model_win, f)

# Saving the Isolation Forest baseline
with open('../models/can_isolation_forest.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)

print("CAN models saved successfully to the models/ directory!")

Saving CAN IDS models as .pkl for final deliverable...


CAN models saved successfully to the models/ directory!
